# Modelagem — Score de detração pós-viagem

Este notebook treina e avalia o modelo que estima, para cada jornada encerrada, a probabilidade de o Cliente se tornar Detrator. Ele começa onde o `pre-processamento.ipynb` termina: consome a base analítica já integrada e não repete nenhuma etapa de limpeza ou integração.

Princípios adotados:

- consumir a base analítica pronta, sem reprocessar as fontes brutas;
- importar a lógica de seleção, divisão e transformação de `scripts/preprocessamento_nps.py`, nunca reescrevê-la aqui;
- ajustar imputação, codificação e escala somente no treino;
- manter o conjunto de teste intocado até a etapa de avaliação;
- versionar o notebook sem saídas de execução e sem nenhum dado do parceiro.

O notebook é preenchido por etapas, cada uma com card próprio no board:

| Seção | Conteúdo | Card |
|---|---|---|
| 1 | Contrato de entrada e carga da matriz | #100 |
| 2 | Baselines de referência | #101 |
| 3 | Validação cruzada por Cliente | #102 |
| 4 | Primeiro modelo candidato | #103 |
| 5 | Ajuste de hiperparâmetros | #104 |
| 6 | Métricas de ordenação | #105 |
| 7 | Limiar operacional e matriz de confusão | #106 |
| 8 | Curvas ROC e precisão contra cobertura | #107 |
| 9 | Tabela comparativa | #108 |


## 1. Contrato de entrada

Esta seção fixa o que a etapa de carga e organização dos dados precisa entregar para que a modelagem rode. O contrato existe para que as duas frentes avancem em paralelo: enquanto a organização não estiver pronta, basta que o arquivo abaixo exista com estas colunas para que todo o restante do notebook funcione sem alteração.

**Arquivo esperado:** `data/processed/base_analitica.parquet`, produzido pela seção 13 do `notebooks/pre-processamento.ipynb`. No Colab, o mesmo arquivo pode vir do Drive; a busca de caminho está na célula seguinte e não exige edição de código.

**Colunas obrigatórias.** A base precisa carregar três grupos:

| Grupo | Colunas | Para que serve |
|---|---|---|
| Alvo | `DETRATOR` | variável a ser predita, binária |
| Chaves de partição | `DATA_STD_CONVERTIDA`, `ID_GOLDENRECORD` | corte temporal e agrupamento por Cliente |
| Features | as 11 de `FEATURES_SCORE_POS_VIAGEM` | matriz do modelo |

As 11 features são a allowlist do score pós-viagem definida no módulo de pré-processamento e não são reescritas aqui: `PERFIL_TUDOAZUL`, `VOO_TIPO`, `TIPO_ENTRETENIMENTO`, `CANAL_COMPRA`, `SEGMENTO`, `ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA`, `CANCELAMENTO_VOO`, `ANTECEDENCIA_CANCELAMENTO`, `TEMPO_VOO`, `QTDE_VIAGENS_12M`. Qualquer mudança nessa lista é decisão de modelagem e acontece no módulo, não no notebook.

**Índices das partições.** A divisão não é aleatória. `dividir_treino_teste_temporal_por_cliente` reserva os meses finais para teste e retira do treino todo Cliente que aparece no teste, porque a base tem dependência intracliente e um Cliente presente nos dois lados inflaria o desempenho estimado. A função devolve os índices do pandas, e é por eles que treino e teste são referenciados no restante do notebook, nunca por posição.

**O que não pode entrar na matriz do modelo.** Nenhuma coluna iniciada por `NPS_` ou `SUB_`. São respostas coletadas no mesmo instrumento que origina o alvo, e o contrato temporal da seção 4.2.3 da documentação as exclui do modelo.

A proibição vale para a matriz, e não para a base analítica. A base carrega essas colunas por construção: `DETRATOR` é derivado de `NPS_PRINCIPAL` dentro da própria `preparar_base_analitica`, e a persistência em Parquet da seção 13 do pré-processamento não descarta coluna alguma. Exigir uma base sem elas faria o notebook falhar sobre uma entrada correta. Por isso as duas verificações têm papéis distintos, e isso é deliberado:

| Célula | Papel | Interrompe a execução |
|---|---|---|
| 1.3 | diagnóstico da base analítica: confere colunas obrigatórias e features da allowlist, e apenas relata quantas colunas de pesquisa vieram junto | sim para coluna obrigatória ausente, não para coluna de pesquisa presente |
| 1.4 | contrato da matriz do modelo: confere que o conjunto de features é exatamente a allowlist e que nenhuma coluna de pesquisa entrou, na entrada e na saída do pré-processador | sim |


### 1.1. Dependências e módulo de pré-processamento

In [ ]:
!pip install -q pandas numpy scikit-learn pyarrow matplotlib

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Inclui a pasta do módulo tanto no projeto local quanto no Colab. O break evita
# que duas cópias de preprocessamento_nps.py fiquem no caminho ao mesmo tempo,
# o que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import (
    FEATURES_SCORE_POS_VIAGEM,
    validar_parquet,
)

SEMENTE = 42
np.random.seed(SEMENTE)

# O contrato da seção 1 ancora em 11 features. Uma mudança silenciosa na
# allowlist do módulo precisa falhar aqui, e não apenas aparecer no print.
QUANTIDADE_FEATURES_ESPERADA = 11
if len(FEATURES_SCORE_POS_VIAGEM) != QUANTIDADE_FEATURES_ESPERADA:
    raise AssertionError(
        f"A allowlist mudou: {len(FEATURES_SCORE_POS_VIAGEM)} features, "
        f"esperadas {QUANTIDADE_FEATURES_ESPERADA}. "
        "Atualize o contrato de entrada da seção 1 antes de seguir."
    )

print("features do score pós-viagem:", len(FEATURES_SCORE_POS_VIAGEM))


### 1.2. Localização e carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a célula falha com a instrução do que fazer, em vez de seguir com uma base vazia.


In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)

if caminho_base is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Gere-a com a seção 13 de notebooks/pre-processamento.ipynb "
        "ou monte o Drive e ajuste CAMINHOS_POSSIVEIS. Caminhos testados: "
        + ", ".join(str(caminho) for caminho in CAMINHOS_POSSIVEIS)
    )

validar_parquet(caminho_base)
df = pd.read_parquet(caminho_base)

print("base analítica:", caminho_base)
print("linhas:", f"{len(df):,}".replace(",", "."), "| colunas:", df.shape[1])


### 1.3. Diagnóstico da base analítica

A célula confere o que a base precisa ter e interrompe a execução se faltar coluna obrigatória ou feature da allowlist. Falhar aqui é barato; descobrir a ausência depois de treinar não é.

A contagem de colunas `NPS_` e `SUB_` é diagnóstico e não motivo de parada, porque a base analítica carrega essas colunas por construção, como explicado na seção 1. O que não pode acontecer é elas alcançarem a matriz do modelo, e é a seção 1.4 que bloqueia isso.


In [ ]:
COLUNAS_OBRIGATORIAS = {"DETRATOR", "DATA_STD_CONVERTIDA", "ID_GOLDENRECORD"}

faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
if faltando:
    raise KeyError(f"A base não atende ao contrato de entrada. Colunas ausentes: {sorted(faltando)}")

features_ausentes = [coluna for coluna in FEATURES_SCORE_POS_VIAGEM if coluna not in df.columns]
if features_ausentes:
    raise KeyError(f"Features da allowlist ausentes na base: {features_ausentes}")

# Diagnóstico, e não trava: a base analítica carrega as respostas da pesquisa
# por construção, já que DETRATOR é derivado de NPS_PRINCIPAL. O bloqueio
# efetivo acontece sobre a matriz do modelo, na seção 1.4.
colunas_pesquisa = [coluna for coluna in df.columns if coluna.startswith(("NPS_", "SUB_"))]
print("colunas de pesquisa na base analítica:", len(colunas_pesquisa), "(diagnóstico, não bloqueia)")
print("prevalência de Detrator:", f"{df['DETRATOR'].mean():.2%}")


### 1.4. Matriz de modelagem e contrato da matriz

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam
acontecer nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features e
ajustar o pré-processador **apenas no treino**. Inverter as duas últimas seria inofensivo;
ajustar o pré-processador antes de particionar não, porque a mediana do treino passaria a
carregar informação de 2026 e a métrica de teste mediria um modelo que já viu o período que
deveria prever.

As datas de corte não são escolhidas aqui. Elas vêm do registro de decisão em
`documents/extras/politica-de-particionamento-temporal.md`, que é a fonte única: repeti-las como
constante do notebook criaria uma segunda verdade que poderia divergir da documentada.

`sem_data="treino"` envia ao treino as 120.000 linhas sem data, e isso só é defensável porque
`verificar_anterioridade_sem_data` prova, pela ordem de registro, que elas antecedem o período
datado. A verificação roda dentro de `preparar_matriz` e interrompe a execução se deixar de valer.

O contrato da matriz é cobrado nas duas pontas do pré-processador. Na entrada, pega uma allowlist
alterada; na saída, uma transformação que reintroduza uma coluna de pesquisa sob outro nome.
Qualquer uma das duas interrompe a execução.


In [ ]:
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

# Datas do registro de decisão do #127, não escolhidas aqui.
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(
    df,
    corte_validacao=CORTE_VALIDACAO,
    corte_teste=CORTE_TESTE,
    sem_data="treino",
)

x_treino, x_validacao, x_teste = (preparo["x"][k] for k in ("treino", "validacao", "teste"))
y_treino, y_validacao, y_teste = (preparo["y"][k] for k in ("treino", "validacao", "teste"))
grupos_treino = preparo["grupos"]["treino"]
preprocessador = preparo["preprocessador"]
x_treino_transformado = preparo["matrizes"]["treino"]
x_validacao_transformado = preparo["matrizes"]["validacao"]
x_teste_transformado = preparo["matrizes"]["teste"]
numericas = preparo["metadados"]["numericas"]
categoricas = preparo["metadados"]["categoricas"]
metadados_divisao = preparo["metadados"]

display(resumo_da_matriz(preparo))

anterioridade = metadados_divisao["anterioridade_sem_data"]
print("linhas sem data enviadas ao treino:", f"{anterioridade['linhas_sem_data']:,}".replace(",", "."))
print("  correlação de Spearman entre ordem de registro e data:", anterioridade["correlacao_spearman"])
print("  maior id sem data:", anterioridade["maior_id_sem_data"],
      "< menor id datado:", anterioridade["menor_id_datado"])
print("removidas por Cliente recorrente:",
      f"{metadados_divisao['linhas_removidas_por_recorrencia']:,}".replace(",", "."))
print("features:", len(metadados_divisao["features_usadas"]),
      "| numéricas:", len(numericas), "| categóricas:", len(categoricas),
      "| colunas da matriz:", metadados_divisao["colunas_da_matriz"])


## 2. Baselines de referência

Regra da classe majoritária e regressão logística, os dois pisos contra os quais o modelo candidato é julgado.

*A preencher no card #101.*


## 3. Validação cruzada por Cliente

`GroupKFold` de 5 folds dentro do treino, agrupando por `ID_GOLDENRECORD` para que nenhum Cliente apareça em dois folds.

*A preencher no card #102.*


## 4. Primeiro modelo candidato

Classificador de gradient boosting sobre árvores, escolha justificada pela ausência de preditor dominante e pela interação entre fidelização e falha operacional documentadas na seção 4.2.1.

*A preencher no card #103.*


## 5. Ajuste de hiperparâmetros

Busca sobre taxa de aprendizado, profundidade e número de iterações, avaliada apenas nos folds de validação.

*A preencher no card #104.*


## 6. Métricas de ordenação do score

Precisão média, área sob a curva ROC e escore de Brier, calculadas sobre o conjunto de teste.

*A preencher no card #105.*


## 7. Limiar operacional e matriz de confusão

Ponto de corte derivado da capacidade diária de contato da equipe de Experiência do Cliente, e não fixado em 0,5 por omissão.

*A preencher no card #106.*


## 8. Curvas ROC e de precisão contra cobertura

Figuras salvas em `assets/`, no mesmo padrão das da seção 4.2.

*A preencher no card #107.*


## 9. Tabela comparativa

Baselines e modelo candidato nas mesmas métricas e sobre a mesma partição de teste.

*A preencher no card #108.*
